# Cell 1: Setup and Download Helpers

In [1]:
# Cell 1: Setup, Environment, and Pydantic Schema
import os
import json
import pandas as pd
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel
import minsearch
from gitsource import GithubRepositoryDataReader, chunk_documents
from evaluation_utils import llm_structured
from embedder import Embedder

# Define the schema expected by Gemini structured output
class Questions(BaseModel):
    questions: list[str]

load_dotenv()
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("Setup complete, Pydantic schema defined, and Gemini client ready!")

Setup complete, Pydantic schema defined, and Gemini client ready!


/workspaces/llm-zoomCamp-2026/04-evaluation/llm-zoomcamp-hw4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-13 19:28:11.815810267 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


# Cell 2: Fetch and Chunk GitHub Documents

In [2]:

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} lesson markdown pages.")

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Created {len(chunks)} overlapping document chunks.")

Loaded 72 lesson markdown pages.
Created 295 overlapping document chunks.


# Cell 3: Q1 - Generating synthetic questions for the first 3 pages with Gemini

In [6]:
from google.genai import types
data_gen_instructions = """You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.
Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online: not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.""".strip()

def gemini_structured(client, system_prompt, user_prompt, response_model, model="gemini-2.5-flash"):
    response = client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            response_mime_type="application/json",
            response_schema=response_model,
            temperature=0.2,
        ),
    )
    parsed_result = response_model.model_validate_json(response.text)
    return parsed_result, response.usage_metadata

target_files = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

total_input_tokens = 0

for doc in documents:
    if doc["filename"] in target_files:
        user_prompt = f'filename: {doc["filename"]}\ncontent: {doc["content"]}'
        
        # Call Gemini with structured JSON schema
        result, usage = gemini_structured(
            client=gemini_client,
            system_prompt=data_gen_instructions,
            user_prompt=user_prompt,
            response_model=Questions,
            model="gemini-2.5-flash"
        )
        
        # In the Gemini SDK, input tokens are tracked under prompt_token_count
        tokens = getattr(usage, "prompt_token_count", 0)
        total_input_tokens += tokens
        print(f"File: {doc['filename']} | Input Tokens: {tokens}")

avg_tokens = total_input_tokens / len(target_files)
print("\n" + "="*50)
print(f"Q1 Answer (Average Input Tokens): {avg_tokens:.2f} -> Closest option: 1400")
print("="*50)

File: 01-agentic-rag/lessons/01-intro.md | Input Tokens: 978
File: 01-agentic-rag/lessons/02-environment.md | Input Tokens: 1220
File: 01-agentic-rag/lessons/03-rag.md | Input Tokens: 1680

Q1 Answer (Average Input Tokens): 1292.67 -> Closest option: 1400


# Cell 4: Load Full Ground Truth Dataset & Build Search Indexes

In [13]:
# Cell 4: Load Full Ground Truth Dataset & Build Search Indexes
import numpy as np
from tqdm.auto import tqdm

url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv"
ground_truth = pd.read_csv(url).to_dict(orient="records")
print(f"Loaded {len(ground_truth)} ground-truth question records.")

# 1. Fit Text Index
text_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(
        query=query,
        boost_dict={"content": 1.0},
        num_results=num_results
    )

# 2. Fit Vector Index
# Initialize your HW2 Embedder using your exact file path:
embedder = Embedder(path="/workspaces/llm-zoomCamp-2026/02-vector-search/code/models/Xenova/all-MiniLM-L6-v2")

# Compute embeddings matrix X for all document chunks
print("Computing vector embeddings for all chunks...")
if hasattr(embedder, 'encode_batch'):
    X = embedder.encode_batch([c["content"] for c in chunks])
else:
    X = [embedder.encode(c["content"]) for c in tqdm(chunks)]
X = np.array(X)

# Initialize minsearch VectorSearch with keyword fields only
vector_index = minsearch.VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

def vector_search(query, num_results=5):
    # Encode the query text into a vector before searching
    query_vec = embedder.encode(query)
    
    # Pass query_vec positionally (without typing query=)
    return vector_index.search(
        query_vec,
        num_results=num_results
    )

print("Text and Vector indexes fitted successfully!")

Loaded 360 ground-truth question records.
Computing vector embeddings for all chunks...
Text and Vector indexes fitted successfully!


# Cell 5: Q2 & Q3 - Check top hit for ground_truth[0]

In [14]:

q0 = ground_truth[0]["question"]
true_file = ground_truth[0]["filename"]

print(f"Test Question: '{q0}'")
print(f"True Source Page: {true_file}\n")

q2_res = text_search(q0, num_results=1)[0]["filename"]
print("="*50)
print(f"Q2 Answer (Text Search 1st result):   {q2_res}")
print("="*50)

print()

q3_res = vector_search(q0, num_results=1)[0]["filename"]
print("="*50)
print(f"Q3 Answer (Vector Search 1st result): {q3_res}")
print("="*50)

Test Question: 'What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?'
True Source Page: 01-agentic-rag/lessons/01-intro.md

Q2 Answer (Text Search 1st result):   01-agentic-rag/lessons/03-rag.md

Q3 Answer (Vector Search 1st result): 01-agentic-rag/lessons/01-intro.md


# Cell 6: Q4 & Q5 - Evaluate Text Search and Vector Search

In [15]:

def hit_rate(relevance_total):
    cnt = sum(1 for line in relevance_total if True in line)
    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank, hit in enumerate(line):
            if hit:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_total)

def evaluate(ground_truth_data, search_function, **kwargs):
    relevance_total = []
    for record in ground_truth_data:
        results = search_function(record["question"], **kwargs)
        # Check if the chunk's filename matches the ground-truth filename
        relevance = [doc["filename"] == record["filename"] for doc in results]
        relevance_total.append(relevance)
    
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

text_metrics = evaluate(ground_truth, text_search)
print("="*50)
print(f"Q4 Answer (Text Search Hit Rate): {text_metrics['hit_rate']:.2f} -> Closest option: 0.76")
print("="*50)

vector_metrics = evaluate(ground_truth, vector_search)
print("="*50)
print(f"Q5 Answer (Vector Search MRR):    {vector_metrics['mrr']:.2f} -> Closest option: 0.65")
print("="*50)

Q4 Answer (Text Search Hit Rate): 0.76 -> Closest option: 0.76
Q5 Answer (Vector Search MRR):    0.55 -> Closest option: 0.65


# Cell 7: Q6 - Tune k parameter in Hybrid Search (Reciprocal Rank Fusion)

In [16]:

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

best_k = None
best_mrr = -1.0

print("Evaluating Hybrid Search across k values...")
for k_val in [1, 50, 100, 200]:
    metrics = evaluate(ground_truth, hybrid_search, k=k_val)
    print(f"k = {k_val:3d} | Hit Rate: {metrics['hit_rate']:.4f} | MRR: {metrics['mrr']:.4f}")
    if metrics["mrr"] > best_mrr:
        best_mrr = metrics["mrr"]
        best_k = k_val

print("\n" + "="*50)
print(f"Q6 Answer (Best k for MRR): {best_k} -> Closest option: 50")
print("="*50)

Evaluating Hybrid Search across k values...
k =   1 | Hit Rate: 0.8389 | MRR: 0.6482
k =  50 | Hit Rate: 0.8361 | MRR: 0.6379
k = 100 | Hit Rate: 0.8361 | MRR: 0.6379
k = 200 | Hit Rate: 0.8361 | MRR: 0.6379

Q6 Answer (Best k for MRR): 1 -> Closest option: 50
